# The dimension of `supp rho_L` on a lattice patch

## The objects, named

* A **Gaussian state** of Majorana modes $\gamma_a$ (Hermitian, $\gamma_a^2 = 1$,
  $\gamma_a\gamma_b = -\gamma_b\gamma_a$) is fixed by its **covariance**
  $\Gamma_{ab} = \tfrac{i}{2}\langle[\gamma_a,\gamma_b]\rangle$, a real antisymmetric matrix.
  The code labels the Majoranas by integers, and site $r$ of a lattice owns the labels $2r$
  and $2r+1$ everywhere below.
* A **patch** $L$ is a set of sites; keeping modes $\in L$ and tracing out everything else $\bar{L}$ leaves the  **patch state** $\rho_L=\mathrm{\mathop{tr}}_{\bar{L}}\ket{\Gamma}\bra{\Gamma}$, whose **block** $\Gamma_L$ is the submatrix of $\Gamma$ on the
  patch's Majoranas.
* The **Williamson decomposition** of the block writes
  $ \Gamma_L = \bigoplus_{k}R^{T}\mathrm{diag}(\lambda_k J)R $ with
  $J = \begin{bmatrix}0&-1\\1&0\end{bmatrix}$, $R$ orthogonal and $0 \leq \lambda_k \leq 1$.
  Mode $k$ is **pure** when $\lambda_k = 1$ and **mixed** otherwise.  A pure mode contributes
  one dimension to $\mathrm{supp}\,\rho_L$ and a mixed mode two, so with a **tolerance**
  $\epsilon$: 
  $$\dim \mathrm{supp}\,\rho_L = 2^{\#\{\lambda_k < 1 - \epsilon\}}.$$
  The exponent $\#\{\lambda_k < 1 - \epsilon\}$ is what this notebook measures.
* $S$ is the von Neumann entropy of the patch state, $-\mathrm{Tr}\,\rho_L\ln\rho_L$
  (`linalg.entropy`, with $k_B=1$).

## What are the pure states made of

For a pure global state $\Gamma = \begin{pmatrix}\Gamma_L & C\\- C^T&\Gamma_{\bar{L}} \end{pmatrix}$, block matrix arguments gives 

$$\Gamma_L^2 + 1 = C\,C^{T},\qquad C = \Gamma_{L,\bar{L}},$$

so $\#\{\lambda_k < 1\} = \tfrac12\operatorname{rank} C$ exactly, and $C$ -- the **cross
covariance** between the patch and everything outside it -- is small exactly where the patch
couples weakly to its outside, i.e. away from the boundary of the patch.  The rank is
therefore maximal in exact arithmetic; what a finite tolerance resolves is a layer at the
boundary, of a depth this notebook measures against $\epsilon$ rather than assumes.


In [ ]:
import math
from dataclasses import dataclass
from typing import Callable

import matplotlib.pyplot as plt
import torch
from torch import Tensor

from FreeFermion.GfPEPS import GfPEPS
from FreeFermion.geometry import Geometry, honeycomb, square, triangular
from FreeFermion.linalg import entropy, is_pure, occupations
from base import REAL

TOLERANCES = (1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12)
QUOTED = 1e-8          # the tolerance every count in the tables is quoted at


def mixed_count(covariance: Tensor, tolerance: float = QUOTED) -> int:
    r'''
    The count of a block at one tolerance: the number of modes with $\lambda_k < 1 - \epsilon$, which are the occupations $p_k = (1-\lambda_k)/2$ above $\epsilon/2$.
    Args:
        covariance: real antisymmetric covariance of the patch, shape (2n, 2n).
        tolerance: the tolerance on $1 - \lambda_k$.
    Returns:
        The count.
    '''
    return int((occupations(covariance) > tolerance / 2).sum())


def num_pure(covariance: Tensor,
              tolerances: tuple[float, ...] = TOLERANCES) -> dict[float, int]:
    '''
    The number of pure modes of a block at every tolerance, keyed by the tolerance, for the tables and the depth plot that need the whole curve.
    Args:
        covariance: real antisymmetric covariance of the patch.
        tolerances: the tolerances.
    Returns:
        tolerance -> count.
    '''
    p = occupations(covariance)
    return {tolerance: int((p > tolerance / 2).sum()) for tolerance in tolerances}


def patch_covariance(covariance: Tensor, patch: list[int]) -> Tensor:
    '''
    The block of a patch, site r owning the Majoranas 2r and 2r+1.
    Args:
        covariance: real antisymmetric covariance of the whole state.
        patch: site labels of the patch.
    Returns:
        The block, shape (2 * len(patch), 2 * len(patch)).
    '''
    majoranas = sorted(2 * site + odd for site in patch for odd in (0, 1))
    return covariance[majoranas][:, majoranas]

## 1. One dimension: a block of a ring

The Kitaev chain on a ring of $N$ sites, site $j$ owning the Majoranas $2j$ and $2j+1$,
$$H = -w\sum_j\left(c_j^\dagger c_{j+1} + \mathrm{h.c.}\right)
     -\mu\sum_j\left(c_j^\dagger c_j - \tfrac12\right)
     +\Delta\sum_j\left(c_j c_{j+1} + \mathrm{h.c.}\right).$$
`kitaev_terms` writes the terms of that Hamiltonian, `majorana_matrix` collects them into the
real antisymmetric matrix $M$ of $H = (i/4)\sum_{ab}M_{ab}\gamma_a\gamma_b$, and
`GfPEPS.from_hamiltonian` returns the ground state covariance of $M$ through its sign
function.  The patch is a contiguous block of $L$ sites, the first $L$ of the ring.  `flux`
flips the sign of the wrap bond, which puts the ground state in the antiperiodic sector: the
critical Ising ring, against the gapped rings of the other parameters.

In [ ]:
def majorana_matrix(terms: list[tuple[int, int, complex]], num_majoranas: int,
                    device: torch.device = torch.device('cuda')) -> Tensor:
    r'''
    Assemble the real antisymmetric $M$ of $H = (i/4)\gamma^T M \gamma$ from the terms
    $\alpha\,\gamma_a\gamma_b$, with every $\alpha$ purely imaginary.
    Args:
        terms: (a, b, alpha) contributions, summed over repeated pairs.
        num_majoranas: dimension of M.
        device: device of the returned matrix.
    Returns:
        The real antisymmetric matrix M.
    Raises:
        ValueError: if a pair repeats a Majorana, or an alpha is not purely imaginary.
    '''
    entries: dict[tuple[int, int], float] = {}
    for a, b, alpha in terms:
        if a == b:
            raise ValueError(f'the pair ({a}, {b}) repeats a Majorana')
        if abs(alpha.real) > 1e-15:
            raise ValueError(f'alpha {alpha} of ({a}, {b}) is not purely imaginary')
        if a > b:
            a, b, alpha = b, a, -alpha
        coefficient = -2j * alpha
        if abs(coefficient.imag) > 1e-15:
            raise ValueError(f'alpha {alpha} of ({a}, {b}) gives a complex M entry')
        entries[(a, b)] = entries.get((a, b), 0.0) + coefficient.real
    matrix = torch.zeros((num_majoranas, num_majoranas), dtype=REAL, device=device)
    index = torch.tensor(list(entries), dtype=torch.long, device=device).reshape(-1, 2)
    values = torch.tensor(list(entries.values()), dtype=REAL, device=device)
    matrix[index[:, 0], index[:, 1]] = values
    return matrix - matrix.T


def kitaev_terms(num_sites: int, w: float = 1.0, delta: float = 1.0, mu: float = 0.0,
                 flux: bool = False) -> list[tuple[int, int, complex]]:
    r'''
    Terms of the Kitaev chain on a ring, site j on the Majoranas (2j, 2j+1).
    Args:
        num_sites: number of sites.
        w: hopping.
        delta: pairing.
        mu: chemical potential.
        flux: negate the wrap bond, i.e. antiperiodic boundary conditions.
    Returns:
        The (a, b, alpha) terms.
    '''
    terms = []
    for j in range(num_sites):
        jp = (j + 1) % num_sites
        sign = -1.0 if flux and j == num_sites - 1 else 1.0
        terms += [(2 * j, 2 * j + 1, -0.5j * mu),
                  (2 * j, 2 * jp + 1, sign * (-0.5j * w)),
                  (2 * jp, 2 * j + 1, sign * (-0.5j * w)),
                  (2 * j, 2 * jp + 1, sign * (0.5j * delta)),
                  (2 * jp, 2 * j + 1, sign * (-0.5j * delta))]
    return terms


NUM_SITES = 600
LENGTHS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 200]
RINGS = {r'\mu = 1': (1.0, False), r'\mu = 1.9': (1.9, False),
         r'\mu = 3.0': (3.0, False), r'\mu = 2.0,  critical, antiperiodic': (2.0, True)}
blocks = {}
for label, (mu, flux) in RINGS.items():
    covariance = GfPEPS.from_hamiltonian(
        majorana_matrix(kitaev_terms(NUM_SITES, mu=mu, flux=flux), 2 * NUM_SITES)).tensors[0]
    blocks[label] = {length: patch_covariance(covariance, list(range(length)))
                     for length in LENGTHS}

print(f'{"ring":>26} {"S(200)":>8} '
      + ' '.join(f'{t:>6.0e}' for t in TOLERANCES[1::2]))
for label, curves in blocks.items():
    print(f'{label:>26} {entropy(curves[200]):>8.4f} '
          + ' '.join(f'{num_pure(curves[200])[t]:>6}' for t in TOLERANCES[1::2]))

figure, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
for label, curves in blocks.items():
    axes[0].plot(LENGTHS, [mixed_count(curves[length]) for length in LENGTHS],
                 marker='.', label=rf'${label}$')
    axes[1].plot(LENGTHS, [entropy(curves[length]) for length in LENGTHS],
                 marker='.', label=rf'${label}$')
axes[0].plot([1, 200], [1, 200], 'k--', lw=1, label=r'$\#\mathrm{mixed} = L$')
axes[0].set(xscale='log', yscale='log', xlabel='$L$',
            ylabel=r'$\#\{\lambda_k < 1 - 10^{-8}\}$',
            title=r'the resolvable $\log_2\dim\mathrm{Im}\,\rho_L$')
axes[1].set(xscale='log', xlabel='$L$', ylabel='$S$', title='entropy')
for axis in axes:
    axis.legend(fontsize=7)
figure.tight_layout()
plt.show()